In [ ]:
"""Cover fractions and 20% labels for every 1000, 500 and 250 px tile.

Reads the "tiles" and "rhodo" layers from labelling.gpkg and writes
tiles_1000px.gpkg, tiles_500px.gpkg and tiles_250px.gpkg.

Usage: python cover_fractions.py labelling.gpkg
Close QGIS (or save all edits) before running.
"""
import sys

import geopandas as gpd
from shapely.geometry import box

LABELLING_PATH = "../Grids/labelling.gpkg"
COVER_THRESHOLD = 0.20                   # cover >= 20% -> positive
LABELLED_STATUSES = ["done", "unsure"]   # "skip" and unlabelled tiles are left out
SIZES = [(1000, 1), (500, 2), (250, 4)]  # (tile size in px, splits per side)

tiles = gpd.read_file(LABELLING_PATH, layer="tiles")
rhododendron_outlines = gpd.read_file(LABELLING_PATH, layer="rhododendron_outlines")
rhododendron_outlines["geometry"] = rhododendron_outlines.make_valid()
rhodo_area = rhododendron_outlines.union_all()           # merge overlaps so no area counts twice

labelled = tiles[tiles["status"].isin(LABELLED_STATUSES)]
print(f"{len(labelled)} labelled tiles, {(tiles['status'] == 'skip').sum()} skipped, "
      f"{tiles['status'].isna().sum()} with no status")

keep = [c for c in tiles.columns if c != "geometry"]
rows = {size: [] for size, _ in SIZES}
for tile in labelled.itertuples(index=False):
    min_x, min_y, max_x, max_y = tile.geometry.bounds
    rhodo_in_tile = rhodo_area.intersection(tile.geometry)   # clip once per tile
    for size, splits in SIZES:
        step_x, step_y = (max_x - min_x) / splits, (max_y - min_y) / splits
        for sub_row in range(splits):                         # row 0 = top
            for sub_col in range(splits):
                sub = box(min_x + sub_col * step_x, max_y - (sub_row + 1) * step_y,
                          min_x + (sub_col + 1) * step_x, max_y - sub_row * step_y)
                cover = rhodo_in_tile.intersection(sub).area / sub.area
                label = int(cover >= COVER_THRESHOLD)
                rows[size].append({**{c: getattr(tile, c) for c in keep},
                                   "tile_size_px": size, "sub_row": sub_row, "sub_col": sub_col,
                                   "cover_fraction": round(cover, 4), "label": label,
                                   "class": "positive" if label else "negative",
                                   "geometry": sub})

for size, _ in SIZES:
    out = gpd.GeoDataFrame(rows[size], geometry="geometry", crs=tiles.crs)
    path = f"../Grids/tiles_{size}px.gpkg"
    out.to_file(path, layer=f"tiles_{size}px", driver="GPKG")
    print(f"{path}: {len(out)} tiles, {out['label'].sum()} positive "
          f"({out['label'].mean():.1%}), mean cover {out['cover_fraction'].mean():.3f}")

In [ ]:
import pandas as pd
fold_of_block = pd.read_csv("../Grids/folds_KD02.csv").set_index("block")["fold"]
for s in [1000, 500, 250]:
    g = gpd.read_file(f"../Grids/tiles_{s}px.gpkg")
    g["fold"] = g["block"].map(fold_of_block)
    g.to_file(f"../Grids/tiles_{s}px.gpkg", layer=f"tiles_{s}px", driver="GPKG")
    print(s, "px: tiles with no fold:", int((g["fold"].isna() & (g["forest"] == "KD02") & (g["role"] == "train")).sum()))

In [ ]:
import geopandas as gpd
t = gpd.read_file("../Grids/tiles_500px.gpkg")
t = t[(t["forest"] == "KD02") & (t["role"] == "train")]
for name, part in [("tiles 1-500", t[t["order"] <= 500]), ("tiles 501-1000", t[t["order"] > 500])]:
    done = part[part["status"] == "done"]
    print(f"{name}: {done['tile_id'].nunique()} done, {part[part['status'] == 'unsure']['tile_id'].nunique()} unsure, "
          f"{int(done['label'].sum())} positive 500 px tiles in {done[done['label'] == 1]['block'].nunique()} blocks")

In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

TIFF_PATH = "../LargeImages/KD02-1-20241203-7_IXM150_mos.tif"
GRID_DIR = Path("../Grids")
OUT_DIR = Path("../ImageFiles/InitialComparison")
SPLITS = {1000: 1, 500: 2, 250: 4}       # sub-tiles per side

# Stage A tiles: KD02 training blocks, "done" only, with a fold
parents = gpd.read_file(GRID_DIR / "tiles_1000px.gpkg")
parents = parents[(parents["forest"] == "KD02") & (parents["role"] == "train")
                  & (parents["status"] == "done") & parents["fold"].notna()]
print(f"{len(parents)} tiles")

written, skipped, problems = 0, 0, []
with rasterio.open(TIFF_PATH) as ds:
    for tile in tqdm(parents.itertuples(), total=len(parents)):
        names = {(size, r, c): OUT_DIR / str(size) / f"{tile.tile_id}_{size}_{r}_{c}.png"
                 for size, n in SPLITS.items() for r in range(n) for c in range(n)}
        if all(p.exists() for p in names.values()):        # already cut earlier
            skipped += 1
            continue
        window = from_bounds(*tile.geometry.bounds, transform=ds.transform).round_offsets().round_lengths()
        rgb = ds.read([1, 2, 3], window=window, boundless=True, fill_value=0)   # bands 1-3 only
        rgb = np.moveaxis(rgb, 0, -1)                                            # (rows, cols, 3)
        if rgb.shape[:2] != (1000, 1000):
            problems.append((tile.tile_id, rgb.shape))
            continue
        for (size, r, c), path in names.items():            # row 0 = top, as in the GeoPackages
            Image.fromarray(rgb[r * size:(r + 1) * size, c * size:(c + 1) * size]).save(path)
            written += 1

print(f"Saved {written} new images; {skipped} tiles already cut")
if problems:
    print(f"{len(problems)} tiles not 1000 x 1000 px, skipped:", problems[:10])

In [ ]:
import geopandas as gpd
g = gpd.read_file("../Grids/tiles_500px.gpkg")       # in Colab: f"{PROJECT}/Grids/tiles_500px.gpkg"
use = g[(g["forest"] == "KD02") & (g["role"] == "train") & (g["status"] == "done") & g["fold"].notna()]
print(len(use), "usable 500 px tiles; highest order:", int(use["order"].max()))